In [1]:
import numpy 
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from omegaconf import OmegaConf
import json
import os
__DIR__ = os.path.dirname("../")
import sys
sys.path.append(__DIR__)
from datasets import load_from_disk, concatenate_datasets
import numpy as np
import jsonlines

from tqdm import tqdm

In [2]:
from diff_masking.utils.phi3 import create_masked_phi

In [3]:
import torch

# Set the CUDA device
torch.cuda.set_device(1)  # Replace 1 with the desired CUDA device index

In [4]:
model_name = "microsoft/Phi-3-mini-128k-instruct"
model = AutoModelForCausalLM.from_pretrained( 
            model_name,  
            device_map="cuda:1",  
            torch_dtype=torch.bfloat16,  
            trust_remote_code=True,  
            attn_implementation="flash_attention_2"
) 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
checkpoint = "../outputs/main/2024-10-26_09-47-07/"

In [7]:
with open(checkpoint+"config.json", 'r') as file:
    config_dict = json.load(file)

In [8]:
config = OmegaConf.create(config_dict)

In [9]:
model = create_masked_phi(model, config.specs.target_layers, config.specs.init_prob, config.specs.tau, checkpoint+"model.pth")

/media/data/nicolas/project/mechanisms/KG_networks/notebooks/../diff_masking/utils/phi3.py:384: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mask_parameters = torch.load(ch

In [10]:
model.cuda()
torch.cuda.empty_cache()

In [11]:
for l in config.specs.target_layers:
    model.model.layers._modules[str(l)].self_attn.mask_enabled = True
    model.model.layers._modules[str(l)].mlp.mask_enabled = True

In [12]:
import json

with open('questions/astronomy.json', 'r') as file:
    astronomy_questions = json.load(file)
with open('questions/biology.json', 'r') as file:
    biology_questions = json.load(file)
with open('questions/quantum.json', 'r') as file:
    quantum_questions = json.load(file)

In [13]:
generation_args = { 
        "max_new_tokens": 1024, 
        "temperature": 0.5, 
        "do_sample": True,
} 

In [14]:
responses = []
for k, v in tqdm(biology_questions.items()):
    messages = [[{"role": "user", "content": q["question"]}] for q in v]
    tokenied = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True, padding=True, return_dict=True)
    input_ids = tokenied["input_ids"]
    attn_mask = tokenied["attention_mask"]

    input_ids = input_ids.to(model.device)
    attn_mask = attn_mask.to(model.device)
        
    outputs = model.generate(input_ids=input_ids,attention_mask=attn_mask, **generation_args)
    decoded_texts = tokenizer.batch_decode(outputs[:, input_ids.shape[1]], skip_special_tokens=False)
    responses.extend([{"user": v[i]["question"], "assistant": decoded_texts[i]} for i in range(len(v))])
output_file = "biology.jsonl"

# Open the file in write mode
with jsonlines.open(output_file, mode='w') as writer:
    # Iterate over the list and write each element as a separate line
    for item in responses:
        writer.write(item)

100%|██████████| 5/5 [09:39<00:00, 115.89s/it]


In [15]:
responses = []
for k, v in tqdm(quantum_questions.items()):
    messages = [[{"role": "user", "content": q["question"]}] for q in v]
    tokenied = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True, padding=True, return_dict=True)
    input_ids = tokenied["input_ids"]
    attn_mask = tokenied["attention_mask"]

    input_ids = input_ids.to(model.device)
    attn_mask = attn_mask.to(model.device)
        
    outputs = model.generate(input_ids=input_ids,attention_mask=attn_mask, **generation_args)
    decoded_texts = tokenizer.batch_decode(outputs[:, input_ids.shape[1]], skip_special_tokens=False)
    responses.extend([{"user": v[i]["question"], "assistant": decoded_texts[i]} for i in range(len(v))])
output_file = "quantum.jsonl"

# Open the file in write mode
with jsonlines.open(output_file, mode='w') as writer:
    # Iterate over the list and write each element as a separate line
    for item in responses:
        writer.write(item)

100%|██████████| 5/5 [19:00<00:00, 228.19s/it]


In [ ]:
responses = []
for k, v in tqdm(astronomy_questions.items()):
    messages = [[{"role": "user", "content": q["question"]}] for q in v]
    tokenied = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True, padding=True, return_dict=True)
    input_ids = tokenied["input_ids"]
    attn_mask = tokenied["attention_mask"]

    input_ids = input_ids.to(model.device)
    attn_mask = attn_mask.to(model.device)
        
    outputs = model.generate(input_ids=input_ids,attention_mask=attn_mask, **generation_args)
    decoded_texts = tokenizer.batch_decode(outputs[:, input_ids.shape[1]], skip_special_tokens=False)
    responses.extend([{"user": v[i]["question"], "assistant": decoded_texts[i]} for i in range(len(v))])

output_file = "astronomy.jsonl"

# Open the file in write mode
with jsonlines.open(output_file, mode='w') as writer:
    # Iterate over the list and write each element as a separate line
    for item in responses:
        writer.write(item)

/media/data/nicolas/conda_envs/eureka-slm/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
